## Streaming

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

In [ ]:
# os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
# os.environ["LANGCHAIN_TRACING_V2"] = "true" # Enable LangSmith tracing
# os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' # Set the LangSmith project name

In [48]:
# Creating the Langfuse tracker 
from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler

os.environ["LANGFUSE_SECRET_KEY"] = str(os.getenv("LANGFUSE_SECRET_KEY"))
os.environ["LANGFUSE_PUBLIC_KEY"] = str(os.getenv("LANGFUSE_PUBLIC_KEY"))
os.environ["LANGFUSE_BASE_URL"]=str(os.getenv("LANGFUSE_BASE_URL"))


lf = Langfuse()
print(lf)

langfuse = get_client()
handler = CallbackHandler()

In [64]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

model = ChatOpenAI(
    api_key = API_KEY,
    base_url = BASE_URL,
    model = 'gpt-5.1',
    temperature = 1.4
)

template = ChatPromptTemplate.from_template(template='Roast in about 2 lines: {topic}, give me 5 of such roasts')

chain = template | model | parser
chain = chain.with_config({"callbacks": [handler],"run_name": "roast_chain"})

In [ ]:
result = chain.invoke({'topic': 'girlfrined'})
print(result)

1. “You’re like my phone on 1% — constantly dying on me but somehow still opening new tabs of drama.”  

2. “Being with you is like a software update: takes forever, drains all my energy, and I still don’t see any improvements.”  

3. “You say you’re ‘low maintenance’ but your attitude has more updates than the App Store.”  

4. “You’re not toxic, you’re just a full-time emotional subscription with no cancel button.”  

5. “If mixed signals were a profession, you’d be a CEO with unlimited overtime.”


For streaming we use the stream method of the chain

In [51]:
for chunk in chain.stream({'topic': 'gym'}):
    print(chunk, end='')

1. The gym is just a building where people pay monthly to move things that never needed moving.  
2. Funny how the heaviest thing in the gym is the silence when someone asks, “So, how’s that New Year’s resolution going?”

3. The gym is where people drive 20 minutes to walk on a machine that goes nowhere.  
4. It’s the only place where you can stare at yourself in the mirror for an hour and still not like what you see.

5. The gym is basically adult PE class, except now you pay for the humiliation.  
6. It’s where you learn that your greatest lift is still your excuses.

In [54]:
async for chunk in chain.astream({'topic': 'girlfriend shopping irrationally'}, config = {'callbacks': [handler]}):
    print(chunk, end='')


1. “Babe, watching you shop is like witnessing a financial horror movie—no plot, just random spending and my wallet screaming in the background.”  

2. “You don’t go shopping, you go on rescue missions for items that were never in danger and definitely never on the list.”  

3. “Your shopping cart looks like you blacked out halfway through a TikTok haul and just trusted your impulses to finish the job.”  

4. “You say you’re ‘just browsing’ the same way arsonists say they’re ‘just playing with matches.’ My bank account can feel the flames.”  

5. “You treat the ‘Add to Cart’ button like a stress ball—every minor inconvenience and boom, another package scheduled to ruin my savings.”

In [60]:
async for event in chain.astream_events({"topic":"girlfriend"}, version="v2", config = {'callbacks': [handler]}):
    print(event, flush=True)

{'event': 'on_chain_start', 'data': {'input': {'topic': 'girlfriend'}}, 'name': 'roast_chain', 'tags': [], 'run_id': '019e1bb5-4bb7-74d0-8f40-90072a29bd54', 'metadata': {}, 'parent_ids': []}
{'event': 'on_prompt_start', 'data': {'input': {'topic': 'girlfriend'}}, 'name': 'ChatPromptTemplate', 'tags': ['seq:step:1'], 'run_id': '019e1bb5-4bbf-7c71-af28-05879774e3d9', 'metadata': {}, 'parent_ids': ['019e1bb5-4bb7-74d0-8f40-90072a29bd54']}
{'event': 'on_prompt_end', 'data': {'output': ChatPromptValue(messages=[HumanMessage(content='Roast in about 2 lines: girlfriend, give me 5 of such roasts', additional_kwargs={}, response_metadata={})]), 'input': {'topic': 'girlfriend'}}, 'run_id': '019e1bb5-4bbf-7c71-af28-05879774e3d9', 'name': 'ChatPromptTemplate', 'tags': ['seq:step:1'], 'metadata': {}, 'parent_ids': ['019e1bb5-4bb7-74d0-8f40-90072a29bd54']}
{'event': 'on_chat_model_start', 'data': {'input': {'messages': [[HumanMessage(content='Roast in about 2 lines: girlfriend, give me 5 of such roa

In [61]:
for chunk in chain.stream({'topic': 'work'}, config = {'callbacks': [handler]}):
    print(chunk, end='')

1. Work really said, “What if we took all your free time and still left you broke and tired?”  
2. Whoever invented work definitely lost a bet and made it everyone else’s problem.  

3. Work is just school without recess, naps, or the slightest bit of hope.  
4. It’s impressive how work manages to feel both endless and never worth the paycheck.  

5. Work is like a bad ex: drains your energy, ruins your weekends, and still thinks you’re “like family.”

In [67]:
for token in chain.stream({'topic': 'mental health'}):
    print(token, end = '|')

||1|.| Your| mental| health| so| unstable|,| even| your| coping| mechanisms| need| coping| mechanisms|.|  
|2|.| Your| inner| peace| got| lost| in| your| brain|’s| group| chat| arguments| and| never| came| back|.|  
|3|.| Your| anxiety| refresh|es| scenarios| like| it|’s| trying| to| crash|-test| every| possible| bad| decision|.|  
|4|.| Your| therapist|’s| therapist| probably| has| a| file| with| your| name| on| it| by| now|.|  
|5|.| Your| brain| treats| minor| inconven|iences| like| season| finales| of| a| drama| no| one| else| is| watching|.|||

In [68]:
import asyncio

async def stream_events():
    async for event in chain.astream_events({'topic': 'GenAI'}):
        if event['event'] == 'on_chat_model_start':
            print('StreamStarted....')
        elif event["event"] == "on_chat_model_stream":
            print(event["data"]["chunk"].content, end="", flush=True)

await stream_events()

StreamStarted....
1. “GenAI, write this for me” – bold move asking a calculator to do your taxes and then blaming it when you get audited.  

2. Depending on GenAI for originality is like photocopying a photocopy and wondering why it looks cursed.  

3. People: “GenAI will replace me.”  
   Also people: *copy-pastes prompt, can’t spell ‘judgment’ without autocorrect.*  

4. Using GenAI as your whole personality is like bringing PowerPoint to a stand‑up show and calling it charisma.  

5. “I beat GenAI on this task” is the new “I beat a toaster at making toast” — correct, but why are you bragging?